In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

!pip install optuna
import optuna

from optuna.study import MaxTrialsCallback

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 6.3 MB/s eta 0:00:00


In [3]:
df = pd.read_csv('/content/drive/MyDrive/OnSafe/modeling_final.csv')

In [4]:
# 제외할 키워드 목록
exclude_keywords = ['video', 'file_id', 'frame', 'timestamp']

# 키워드가 하나라도 포함되면 제외
drop_col = [col for col in df.columns if not any(keyword in col.lower() for keyword in exclude_keywords)]

# 컬럼 선택
df = df[drop_col]

df

,neck_angle,neck_angular_velocity,neck_angular_acceleration,shoulder_balance_angle,shoulder_balance_angular_velocity,shoulder_balance_angular_acceleration,shoulder_left_angle,shoulder_left_angular_velocity,shoulder_left_angular_acceleration,shoulder_right_angle,...,spine_angular_acceleration,ankle_left_angle,ankle_left_angular_velocity,ankle_left_angular_acceleration,ankle_right_angle,ankle_right_angular_velocity,ankle_right_angular_acceleration,center_distance,center_speed,Label
0,20.680788,-12.271715,184.552498,117.980466,2498.058166,7052.299677,25.160310,365.426342,4574.281863,43.718651,...,-2055.415069,121.146172,-1041.416422,-5435.284408,134.870221,-1445.213398,-8822.845456,1.942890e-16,1.165734e-14,0.0
1,25.160022,6.151750,-3977.881772,126.860540,235.076656,-74874.669237,32.936111,152.476062,-12993.440283,49.397831,...,-6805.643321,107.223897,-181.176147,40327.731608,116.830968,-294.094849,53051.507406,3.955170e-16,2.373102e-14,0.0
2,20.885847,-144.867774,-425.943202,125.816355,2.235858,-8595.416016,30.242845,-67.688334,-5016.894535,50.575347,...,3083.921846,115.106967,302.841298,7890.131248,125.067060,323.170182,12170.580699,4.437638e-16,2.662583e-14,0.0
3,20.331096,-8.046357,4197.226968,126.935069,-51.437211,1911.131293,30.679833,-14.753756,-542.685302,53.972380,...,10712.091864,117.318607,81.828228,-9271.617768,127.603307,111.591175,-7018.747461,2.087443e-16,1.252466e-14,0.0
4,20.617635,-4.960209,-323.045662,124.101781,65.940234,8833.618161,29.751053,-85.777844,-941.782118,48.057296,...,-438.093245,117.834575,-6.212628,-5679.841082,128.786766,89.211933,-2096.623552,2.498002e-16,1.498801e-14,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296562,25.626388,31.437188,-739.170751,116.499138,-91.135295,1752.576522,14.011173,-2.501825,-726.467353,50.954784,...,-73.892263,116.095380,-24.295745,454.977069,107.489540,-32.471702,-799.993634,4.996004e-16,1.448841e-14,1.0
296563,26.102950,-14.325184,-1129.321421,114.687117,27.905029,3241.283093,13.426289,-28.977733,-756.436352,51.931975,...,11.409774,115.623805,14.651826,1599.422870,106.294996,-32.522526,696.040891,2.498002e-16,7.244205e-15,1.0
296564,24.638445,-46.447048,53.555766,118.423623,132.401470,-598.446070,12.012709,-54.669849,1400.294337,51.446722,...,-529.589874,117.105850,86.009281,375.720456,105.246607,15.531118,633.252007,3.532708e-16,1.024485e-14,1.0
296565,22.899705,-10.631683,7443.498688,123.818253,-13.367114,-12608.788308,9.655955,67.594290,6448.057128,58.201012,...,-9117.846468,121.555480,40.563581,-2798.553521,107.366108,11.150026,-79.421324,3.532708e-16,1.024485e-14,1.0


In [5]:
X = df.drop(columns='Label')
y = df['Label']

In [6]:
# Train/Test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
# 정규화
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns)

In [8]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print("원본 데이터 분포:\n", y.value_counts())
print("SMOTE 적용 후 데이터 분포:\n", y_train_res.value_counts())

원본 데이터 분포:
 Label
1.0    194917
0.0    101650
Name: count, dtype: int64
SMOTE 적용 후 데이터 분포:
 Label
1.0    155933
0.0    155933
Name: count, dtype: int64


In [11]:
def objective(trial):
    params = {
        "n_estimators"     : trial.suggest_int  ("n_estimators",      100, 300),
        "max_depth"        : 5,
        "min_child_weight" : trial.suggest_int  ("min_child_weight",    1,  10),
        "learning_rate"    : trial.suggest_float("learning_rate",    0.01, 0.3, log=True),
        "subsample"        : trial.suggest_float("subsample",         0.6, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree",  0.6, 1.0),
        "gamma"            : trial.suggest_float("gamma",             0.0, 3.0),
        "reg_alpha"        : trial.suggest_float("reg_alpha",         0.0, 1.0),
        "reg_lambda"       : trial.suggest_float("reg_lambda",        0.0, 3.0),
        "scale_pos_weight" : trial.suggest_float("scale_pos_weight",  0.8, 1.2),
        "eval_metric"      : "logloss",
        "random_state"     : 42,
        "n_jobs"           : -1,
    }

    model = XGBClassifier(**params)

    score = cross_val_score(
        model, X_train_res, y_train_res,
        cv=3, scoring='recall'
    ).mean()

    return score


pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5)
study = optuna.create_study(direction='maximize', pruner=pruner)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"최고 Recall 점수 : {study.best_value:.4f}")

[I 2026-05-19 12:28:25,314] A new study created in memory with name: no-name-fdc4de72-810c-4c8c-9b81-4702c3a1266b


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-05-19 12:28:55,531] Trial 0 finished with value: 0.9236915731853985 and parameters: {'n_estimators': 100, 'min_child_weight': 2, 'learning_rate': 0.07576932258196609, 'subsample': 0.8513320877036075, 'colsample_bytree': 0.8071371289532419, 'gamma': 0.24094373523761292, 'reg_alpha': 0.6139184177696763, 'reg_lambda': 1.6962916774601196, 'scale_pos_weight': 1.020011899642146}. Best is trial 0 with value: 0.9236915731853985.
[I 2026-05-19 12:29:26,915] Trial 1 finished with value: 0.9166436601367214 and parameters: {'n_estimators': 135, 'min_child_weight': 2, 'learning_rate': 0.0330368153287629, 'subsample': 0.6147075073452005, 'colsample_bytree': 0.8317271789215525, 'gamma': 2.1118932626560163, 'reg_alpha': 0.33457759278450017, 'reg_lambda': 1.0001278391823585, 'scale_pos_weight': 0.9104657841800575}. Best is trial 0 with value: 0.9236915731853985.
[I 2026-05-19 12:30:10,722] Trial 2 finished with value: 0.9034713209077054 and parameters: {'n_estimators': 186, 'min_child_weight': 

In [14]:
# 1. 최적의 파라미터 가져오기
best_params = study.best_params

# 2. 고정 파라미터 추가 (objective 함수 내 설정과 동일하게)
best_params["eval_metric"] = "logloss"
best_params["random_state"] = 42
best_params["n_jobs"] = -1

# 3. 최종 모델 정의 및 학습
final_clf = XGBClassifier(**best_params)

# SMOTE 된 데이터로 최종 학습
final_clf.fit(X_train_res, y_train_res)

# 4. 예측 및 평가 (기존과 동일)
y_pred_xgb = final_clf.predict(X_test_scaled)

print(f"최적 파라미터: {best_params}")
print("XGBoost 정확도:", accuracy_score(y_test, y_pred_xgb))
print("XGBoost Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("XGBoost Classification Report:\n", classification_report(y_test, y_pred_xgb))

최적 파라미터: {'n_estimators': 298, 'min_child_weight': 6, 'learning_rate': 0.27498459031412303, 'subsample': 0.9752010596803532, 'colsample_bytree': 0.881875712805713, 'gamma': 2.9298935055194715, 'reg_alpha': 0.49260693468738453, 'reg_lambda': 1.562474828948434, 'scale_pos_weight': 1.1839445802263477, 'eval_metric': 'logloss', 'random_state': 42, 'n_jobs': -1}
XGBoost 정확도: 0.9354621168695417
XGBoost Confusion Matrix:
 [[18193  2137]
 [ 1691 37293]]
XGBoost Classification Report:
               precision    recall  f1-score   support

         0.0       0.91      0.89      0.90     20330
         1.0       0.95      0.96      0.95     38984

    accuracy                           0.94     59314
   macro avg       0.93      0.93      0.93     59314
weighted avg       0.94      0.94      0.94     59314



In [ ]:
print(f"**study.best_params : {study.best_params}")

In [15]:
def objective(trial):
    params = {
        "n_estimators"     : trial.suggest_int  ("n_estimators",      100, 300),
        "max_depth"        : 6,
        "min_child_weight" : trial.suggest_int  ("min_child_weight",    1,  10),
        "learning_rate"    : trial.suggest_float("learning_rate",    0.01, 0.3, log=True),
        "subsample"        : trial.suggest_float("subsample",         0.6, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree",  0.6, 1.0),
        "gamma"            : trial.suggest_float("gamma",             0.0, 3.0),
        "reg_alpha"        : trial.suggest_float("reg_alpha",         0.0, 1.0),
        "reg_lambda"       : trial.suggest_float("reg_lambda",        0.0, 3.0),
        "scale_pos_weight" : trial.suggest_float("scale_pos_weight",  0.8, 1.2),
        "eval_metric"      : "logloss",
        "random_state"     : 42,
        "n_jobs"           : -1,
    }

    model = XGBClassifier(**params)

    score = cross_val_score(
        model, X_train_res, y_train_res,
        cv=3, scoring='recall'
    ).mean()

    return score


pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5)
study = optuna.create_study(direction='maximize', pruner=pruner)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"최고 Recall 점수 : {study.best_value:.4f}")

[I 2026-05-19 13:20:31,135] A new study created in memory with name: no-name-bc44921a-527e-46dd-b27f-c7904bc079f3


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-05-19 13:21:26,776] Trial 0 finished with value: 0.9289181857206549 and parameters: {'n_estimators': 176, 'min_child_weight': 3, 'learning_rate': 0.015865875836642566, 'subsample': 0.8025679982514746, 'colsample_bytree': 0.7916487571237132, 'gamma': 0.9901447013242327, 'reg_alpha': 0.20363948222926342, 'reg_lambda': 0.051829751922261136, 'scale_pos_weight': 0.9913189618665919}. Best is trial 0 with value: 0.9289181857206549.
[I 2026-05-19 13:22:00,022] Trial 1 finished with value: 0.9318553544171081 and parameters: {'n_estimators': 116, 'min_child_weight': 7, 'learning_rate': 0.016387404874385943, 'subsample': 0.6304642703609925, 'colsample_bytree': 0.639856775780948, 'gamma': 2.9636826342323332, 'reg_alpha': 0.9154640016845292, 'reg_lambda': 2.4992424259243298, 'scale_pos_weight': 1.17018051959818}. Best is trial 1 with value: 0.9318553544171081.
[I 2026-05-19 13:22:40,289] Trial 2 finished with value: 0.92184463856047 and parameters: {'n_estimators': 139, 'min_child_weight': 

In [16]:
# 1. 최적의 파라미터 가져오기
best_params = study.best_params

# 2. 고정 파라미터 추가 (objective 함수 내 설정과 동일하게)
best_params["eval_metric"] = "logloss"
best_params["random_state"] = 42
best_params["n_jobs"] = -1

# 3. 최종 모델 정의 및 학습
final_clf = XGBClassifier(**best_params)

# SMOTE 된 데이터로 최종 학습
final_clf.fit(X_train_res, y_train_res)

# 4. 예측 및 평가 (기존과 동일)
y_pred_xgb = final_clf.predict(X_test_scaled)

print(f"최적 파라미터: {best_params}")
print("XGBoost 정확도:", accuracy_score(y_test, y_pred_xgb))
print("XGBoost Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("XGBoost Classification Report:\n", classification_report(y_test, y_pred_xgb))

최적 파라미터: {'n_estimators': 247, 'min_child_weight': 1, 'learning_rate': 0.24816589845369308, 'subsample': 0.8573837806356372, 'colsample_bytree': 0.8867813824142943, 'gamma': 2.773851944245745, 'reg_alpha': 0.7980718720682968, 'reg_lambda': 1.1598027717002473, 'scale_pos_weight': 1.1595547680231633, 'eval_metric': 'logloss', 'random_state': 42, 'n_jobs': -1}
XGBoost 정확도: 0.9324274201706174
XGBoost Confusion Matrix:
 [[18111  2219]
 [ 1789 37195]]
XGBoost Classification Report:
               precision    recall  f1-score   support

         0.0       0.91      0.89      0.90     20330
         1.0       0.94      0.95      0.95     38984

    accuracy                           0.93     59314
   macro avg       0.93      0.92      0.92     59314
weighted avg       0.93      0.93      0.93     59314

